In [1]:
import sys
!{sys.executable} -m pip install hdfs

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number, lower, trim, concat_ws, collect_set, lit, when, regexp_replace, length, concat
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

# 1. Configuração da Sessão Spark
builder = SparkSession.builder \
    .appName("Gold_Dim_Filme_Prints_Final") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Definição de Caminhos
silver_path = "hdfs://hdfs-nn:9000/demo/silver"
gold_path = "hdfs://hdfs-nn:9000/demo/gold/dim_filme"

def ler_delta_com_status(path, nome):
    """Lê Delta e imprime se foi lido com sucesso ou se falhou"""
    try:
        df = spark.read.format("delta").load(path)
        # Padronização de colunas
        if "movie_title" in df.columns: df = df.withColumnRenamed("movie_title", "title")
        elif "name" in df.columns: df = df.withColumnRenamed("name", "title")
        if "genres" in df.columns: df = df.withColumnRenamed("genres", "genero")
        elif "genre" in df.columns: df = df.withColumnRenamed("genre", "genero")
        
        if "genero" not in df.columns: 
            df = df.withColumn("genero", lit(None).cast("string"))
        
        # Filtro de descrições longas
        df = df.filter(length(col("title")) < 150)
        
        print(f"Lido com Sucesso: Silver {nome} (Caminho: {path})")
        return df.select("title", "genero")
    except Exception as e:
        print(f"Erro: Silver {nome} (Caminho: {path}) - Erro: {str(e)[:50]}")
        return None

# 2. Leitura dos 5 Datasets Silver
print("\n-> A Iniciar Processo de Leitura Silver")
f1 = ler_delta_com_status(f"{silver_path}/movies_silver", "Movies")
f2 = ler_delta_com_status(f"{silver_path}/boxoffice_silver", "BoxOffice")
f3 = ler_delta_com_status(f"{silver_path}/rotten_tomatoes", "Rotten Tomatoes")
f4 = ler_delta_com_status(f"{silver_path}/titles_delta", "Netflix")
f5 = ler_delta_com_status(f"{silver_path}/silver_titles", "Amazon")

# Unificação de fontes válidas
dfs = [df for df in [f1, f2, f3, f4, f5] if df is not None]
df_union = dfs[0]
for next_df in dfs[1:]: 
    df_union = df_union.union(next_df)

# 3. Limpeza e Tradução de Números (Mantém Título Original como base)
df_clean = df_union \
    .withColumn("t_mod", regexp_replace(trim(col("title")), r"^[^a-zA-Z0-9]+", "")) \
    .withColumn("t_mod", regexp_replace(lower(col("t_mod")), r"\b1\b", "one")) \
    .withColumn("t_mod", regexp_replace(col("t_mod"), r"\b10\b", "ten")) \
    .filter(col("t_mod") != "")

# 4. Agrupamento e Limpeza de Género
df_gold = df_clean.groupBy("t_mod").agg(collect_set("genero").alias("g_list")) \
    .withColumn("title", col("t_mod")) \
    .withColumn("g_raw", lower(concat_ws(", ", col("g_list")))) \
    .withColumn("g_raw", regexp_replace(col("g_raw"), r"[\[\]']", "")) \
    .withColumn("g_raw", regexp_replace(col("g_raw"), r"&", ",")) \
    .withColumn("g_raw", trim(regexp_replace(col("g_raw"), r"\s+", " "))) \
    .withColumn("genero", when((col("g_raw") == "") | (col("g_raw") == ","), lit(None)).otherwise(col("g_raw")))

# 5. Geração de IDs (Prefixado com 'F')
window = Window.orderBy("title")
dim_filme = df_gold.withColumn("num", row_number().over(window)) \
    .withColumn("id_filme", concat(lit("F"), col("num").cast("string"))) \
    .select("id_filme", "title", "genero", "num") # Mantemos 'num' temporariamente para ordenar

# 6. Gravação Integral no HDFS
print(f"\n-> A Gravar Dimensão Final na Camada GOLD")

dim_filme.drop("num").write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)
print("Processo Concluído: Dimensão Filme guardada com sucesso!")

print("\nTOP 20 Primeiros Filmes da Dimensão Gold")
dim_filme.orderBy("num").select("id_filme", "title", "genero").show(20, truncate= 50)

spark.stop()


-> A Iniciar Processo de Leitura Silver
Lido com Sucesso: Silver Movies (Caminho: hdfs://hdfs-nn:9000/demo/silver/movies_silver)
Lido com Sucesso: Silver BoxOffice (Caminho: hdfs://hdfs-nn:9000/demo/silver/boxoffice_silver)
Lido com Sucesso: Silver Rotten Tomatoes (Caminho: hdfs://hdfs-nn:9000/demo/silver/rotten_tomatoes)
Lido com Sucesso: Silver Netflix (Caminho: hdfs://hdfs-nn:9000/demo/silver/titles_delta)
Lido com Sucesso: Silver Amazon (Caminho: hdfs://hdfs-nn:9000/demo/silver/silver_titles)

-> A Gravar Dimensão Final na Camada GOLD
Processo Concluído: Dimensão Filme guardada com sucesso!

TOP 20 Primeiros Filmes da Dimensão Gold
+--------+---------------------------------------------+--------------------------------------------------+
|id_filme|                                        title|                                            genero|
+--------+---------------------------------------------+--------------------------------------------------+
|      F1|                     